# Resume existing ConceptNet-100k run — original revision

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SemanticMap/semgraphex/blob/feature/recursive-lossless-graph-dictionary-v2/notebooks/16_resume_legacy_cn100k_run_colab.ipynb)

Этот notebook **только продолжает** уже существующий run `wishart-grammar-v2-cn100k-l4-01` с его исходным Git revision и execution lineage. Он не включает новый bounded CPU pipeline и нужен, чтобы не потерять уже вычисленные checkpoints.

In [ ]:
REPO_URL='https://github.com/SemanticMap/semgraphex.git'
PINNED_SHA='d70d5133b60d8cd63e8c374501252bf0bc545eac'
REPO_ROOT='/content/semgraphex'
DRIVE_ROOT='/content/drive/MyDrive/SemanticMap/colab/wishart'
RUN_NAME='wishart-grammar-v2-cn100k-l4-01'
DATASET_DRIVE='data/conceptnet_en_100k.tsv'
DEVICE='cuda'
CPU_WORKERS=4
BLAS_THREADS=1


In [ ]:
import json, os, subprocess, sys
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
DRIVE=Path(DRIVE_ROOT)
run=DRIVE/'runs'/RUN_NAME
manifest=json.loads((run/'input.json').read_text())
assert manifest['code_revision']==PINNED_SHA, manifest['code_revision']
assert (manifest.get('execution_options') or {}).get('cpu_workers')==CPU_WORKERS
assert manifest.get('blas_threads')==BLAS_THREADS
assert (manifest.get('device') or {}).get('selected')==DEVICE
print('Resume identity verified:', {k:manifest.get(k) for k in ['code_revision','blas_threads']})
print('Checkpoint:', json.loads((run/'DRIVE_CHECKPOINT.json').read_text()))


In [ ]:
repo=Path(REPO_ROOT)
if not (repo/'.git').exists():
    subprocess.run(['git','clone',REPO_URL,str(repo)],check=True)
else:
    subprocess.run(['git','-C',str(repo),'fetch','origin'],check=True)
subprocess.run(['git','-C',str(repo),'checkout','--detach',PINNED_SHA],check=True)
constraints=repo/'requirements/constraints-colab.txt'
subprocess.run([sys.executable,'-m','pip','install','-q','-c',str(constraints),'-e',f'{repo}[dev,wishart,gpu]'],check=True)
sha=subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()
assert sha==PINNED_SHA
print('Checked out:',sha)


In [ ]:
config=repo/'configs/wishart_conceptnet_grammar_v2.yaml'
cmd=[sys.executable,'-m','semmap_haken.wishart_colab_cli',
     '--config',str(config),'--drive-root',DRIVE_ROOT,
     '--scratch-root','/content/semmap-wishart-grammar-v2',
     '--run-name',RUN_NAME,'--device',DEVICE,
     '--cpu-workers',str(CPU_WORKERS),'--blas-threads',str(BLAS_THREADS),
     '--compare-mode','size_mtime','--dataset-drive',DATASET_DRIVE,'--resume']
print(' '.join(cmd))
subprocess.run(cmd,cwd=repo,check=True)


После завершения старого run запускайте bounded-pipeline эксперимент отдельно из notebook 15. Не меняйте SHA/worker settings этого notebook: это часть resume identity.